# halarc1407 | 4/4/2026
## Stroke Prediction Dataset - Week 5: Prescriptive Analytics
**SDC380 | Final Project Submission**

*(This notebook extends Week 4 - all Week 4 cells for data loading, encoding, train/test split, model training, metrics, and feature importance are retained above this point.)*

## Feature Actionability Discussion

Not all features in a predictive model are equally actionable. Some, like Age, Gender, or Family History of Stroke, are fixed biological or demographic attributes that cannot be changed through intervention. Others, like Body Mass Index (BMI) and Stress Levels, represent modifiable risk factors that can realistically be targeted by clinical programs or public health policy.

**Body Mass Index (BMI):**
BMI is one of the most actionable features in any cardiovascular or cerebrovascular risk model. It can be reduced through dietary changes, increased physical activity, clinical weight management programs, and in more severe cases, pharmacological or surgical interventions. A 10% reduction in BMI is an achievable target within 6-12 months for most patients with obesity through structured lifestyle programs, and is clinically meaningful - it is associated with improvements in blood pressure, insulin resistance, and lipid profiles, all of which independently reduce stroke risk.

**Stress Levels:**
Stress is also modifiable, though less precisely measurable than BMI. Interventions include cognitive-behavioral therapy (CBT), mindfulness-based stress reduction (MBSR), exercise programs, workplace wellness policies, and community mental health resources. A 5% reduction in measured stress levels is a modest and realistic target for structured stress reduction programs.

**Important caveat about this dataset:**
The Stress Levels coefficient in this logistic regression model is -0.0280 (negative), and the BMI coefficient is -0.0150 (also negative). In a standard clinical dataset, both would be expected to be positive (higher BMI and stress increase stroke risk). The negative signs here are a known artifact of this dataset's deliberately decorrelated design, which was noted in Weeks 3 and 4. This means the simulation below will show a slight *increase* in predicted probability after the interventions - the opposite of what would occur in real clinical data. The simulation is methodologically correct given the model; the limitation is in the dataset itself, not the approach.

In [ ]:
# Week 5 - rebuild model from Week 4 (run this cell after all Week 4 cells)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import warnings; warnings.filterwarnings('ignore')

todays_date = '4/4/2026'
student_id  = 'halarc1407'

# Reload and rebuild (identical to Week 4 pipeline)
df = pd.read_excel('Stroke_Prediction_Dataset.xlsx')
cat_cols = ['Gender','Marital Status','Work Type','Residence Type',
            'Smoking Status','Alcohol Intake','Physical Activity',
            'Family History of Stroke','Dietary Habits']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)
df_encoded['Stroke'] = (df_encoded['Diagnosis'] == 'Stroke').astype(int)
df_encoded = df_encoded.drop(columns=['Patient ID','Patient Name','Diagnosis'])
X = df_encoded.drop('Stroke', axis=1)
y = df_encoded['Stroke']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
display(Markdown(f'**Model rebuilt. {student_id} | {todays_date}**'))

In [ ]:
# Step: Simulate interventions on high-risk subset
# Subset: patients with predicted stroke probability > median
X_all_s = scaler.transform(X)
probs_all = lr.predict_proba(X_all_s)[:, 1]
median_prob = np.median(probs_all)

mask_hr = probs_all > median_prob
X_hr = X[mask_hr].copy()
display(Markdown(f'**High-risk subset (prob > {median_prob:.4f}): {mask_hr.sum():,} patients**'))

# Apply interventions
X_int = X_hr.copy()
X_int['Body Mass Index (BMI)'] = X_int['Body Mass Index (BMI)'] * 0.90   # -10%
X_int['Stress Levels']         = X_int['Stress Levels'] * 0.95           # -5%

# Predict before and after
prob_before = lr.predict_proba(scaler.transform(X_hr))[:, 1]
prob_after  = lr.predict_proba(scaler.transform(X_int))[:, 1]

# Build results dataframe
results_df = X_hr[['Body Mass Index (BMI)', 'Stress Levels']].copy()
results_df['Predicted Likelihood of Stroke Before Intervention'] = prob_before
results_df['BMI After Intervention']           = X_int['Body Mass Index (BMI)']
results_df['Stress After Intervention']        = X_int['Stress Levels']
results_df['Predicted Likelihood of Stroke after intervention'] = prob_after
results_df['Delta (Before - After)']           = prob_before - prob_after

display(Markdown(f'**Mean probability BEFORE: {prob_before.mean():.4f}**'))
display(Markdown(f'**Mean probability AFTER:  {prob_after.mean():.4f}**'))
display(Markdown(f'**BMI coeff: {lr.coef_[0][list(X.columns).index("Body Mass Index (BMI)")]:.4f} | '
                 f'Stress coeff: {lr.coef_[0][list(X.columns).index("Stress Levels")]:.4f}**'))
display(results_df.head(10))

In [ ]:
# Scatter plot: before vs after intervention
np.random.seed(42)
idx = np.random.choice(len(prob_before), size=300, replace=False)
pb_s = prob_before[idx]
pa_s = prob_after[idx]

fig, ax = plt.subplots(figsize=(10, 6.5))
lim = [min(pb_s.min(), pa_s.min())-0.01, max(pb_s.max(), pa_s.max())+0.01]
ax.plot(lim, lim, '--', color='#888888', linewidth=1.5, label='No Change (reference)', zorder=1)
ax.scatter(pb_s, pa_s, alpha=0.55, c='#C00000', edgecolors='white',
           linewidths=0.4, s=40, label='Patient (n=300 sample)', zorder=3)
ax.set_xlabel('Predicted Stroke Probability - Before Intervention', fontsize=11, fontweight='bold')
ax.set_ylabel('Predicted Stroke Probability - After Intervention', fontsize=11, fontweight='bold')
ax.set_title(f'Stroke Risk Before vs After Intervention (BMI -10%, Stress -5%)\n{student_id} - {todays_date}',
             fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, loc='upper left')
ax.text(0.97, 0.05,
        f'Subset n={len(prob_before):,}\nMean before: {prob_before.mean():.4f}\n'
        f'Mean after:  {prob_after.mean():.4f}\nBMI coeff: -0.0150\nStress coeff: -0.0280',
        transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#EAF2FB', edgecolor='#2E75B6', alpha=0.9))
plt.tight_layout()
plt.show()